# Day 3: Advanced Python Mini Project
## Simple Task Manager
Run the code cells from top to bottom. Tasks are saved in `tasks.json`.

In [ ]:
# Run once if requests is not installed
# %pip install requests

import json
import logging
from abc import ABC, abstractmethod
from contextlib import contextmanager
from functools import wraps
from pathlib import Path
import requests

DATA_FILE = Path('tasks.json')
logging.basicConfig(filename='task_manager.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Decorator + context manager
def log_action(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        logging.info(f'Running {function.__name__}')
        return function(*args, **kwargs)
    return wrapper

@contextmanager
def open_data_file(mode):
    file = DATA_FILE.open(mode, encoding='utf-8')
    try:
        yield file
    finally:
        file.close()

In [ ]:
# Abstraction, encapsulation, inheritance, and polymorphism
class Task(ABC):
    def __init__(self, title, completed=False):
        self._title = title
        self.completed = completed

    @property
    def title(self):
        return self._title

    @abstractmethod
    def describe(self):
        pass

    def to_dict(self):
        return {'title': self.title, 'completed': self.completed, 'type': type(self).__name__}

class PersonalTask(Task):
    def describe(self):
        return f'Personal: {self.title}'

class StudyTask(Task):
    def __init__(self, title, topic, completed=False):
        super().__init__(title, completed)
        self.topic = topic

    def describe(self):
        return f'Study ({self.topic}): {self.title}'

    def to_dict(self):
        data = super().to_dict()
        data['topic'] = self.topic
        return data

In [ ]:
# Iterator, generator, JSON handling, and logging
class TaskManager:
    def __init__(self):
        self.tasks = []

    def __iter__(self):
        return iter(self.tasks)

    def pending_tasks(self):
        yield from (task for task in self.tasks if not task.completed)

    @log_action
    def add_task(self, task):
        self.tasks.append(task)

    def save(self):
        with open_data_file('w') as file:
            json.dump([task.to_dict() for task in self.tasks], file, indent=2)

    def show_tasks(self):
        for number, task in enumerate(self, start=1):
            status = 'Done' if task.completed else 'Pending'
            print(f'{number}. [{status}] {task.describe()}')

In [ ]:
# Create and display tasks
manager = TaskManager()
manager.add_task(PersonalTask('Buy groceries'))
manager.add_task(StudyTask('Practice classes', 'OOP'))
manager.tasks[0].completed = True
manager.show_tasks()
manager.save()

In [ ]:
# Generator example: show only unfinished tasks
for task in manager.pending_tasks():
    print(task.describe())

In [ ]:
# REST API consumption with requests
try:
    response = requests.get('https://api.quotable.io/random', timeout=5)
    response.raise_for_status()
    quote = response.json()
    print(f"Motivation: {quote['content']} — {quote['author']}")
except requests.RequestException:
    print('Could not connect. Keep learning Python!')